# 🏔️ Semana 11 · Unidad 3 — Colas de Prioridad y Binary Heap

## Información del Curso

| Aspecto | Detalle |
|--------|--------|
| **Universidad** | Universidad de Talca, Chile |
| **Carrera** | Ingeniería Civil en Informática |
| **Semestre** | 2°-3° año |
| **Curso** | Algoritmos y Estructuras de Datos |
| **Docente** | PhD. César Astudillo |
| **Clase** | Semana 11 · Unidad 3 — Priority Queues y Binary Heap |
| **Duración** | 50 minutos |

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.  
> Ejecuta las celdas en orden de arriba hacia abajo.*

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import display, HTML
print("✅ Dependencias cargadas correctamente")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Definir** el ADT Priority Queue y distinguirlo de una cola FIFO.
2. **Representar** un árbol binario completo usando un arreglo con indexación 1-based.
3. **Implementar** las operaciones `swim` y `sink` para mantener el orden del heap.
4. **Implementar** `insert` y `delMax` sobre un max-heap con complejidad O(log n).
5. **Analizar** por qué el heap es más eficiente que alternativas triviales para la Priority Queue.

# Sección 1: ¿Por qué una Priority Queue? (5 minutos)

## El problema que resuelve

En una **cola FIFO**, el orden de salida es el mismo que el de entrada.  
En una **Priority Queue**, el elemento que sale primero es el de **mayor prioridad**, sin importar cuándo llegó.

### Ejemplos concretos

| Aplicación | Elemento | Prioridad |
|-----------|---------|----------|
| Sistema operativo | Proceso | Urgencia / tiempo restante |
| Red hospitalaria | Paciente | Gravedad del caso |
| Simulación de eventos | Evento | Tiempo de ocurrencia |
| Algoritmo de Dijkstra | Nodo | Distancia acumulada mínima |
| Compresión Huffman | Símbolo | Frecuencia de aparición |

> 🎙️ **[PAUSA PROFESOR]** *"¿Cuál sería una Priority Queue en la vida cotidiana? Piensen en situaciones donde 'el que llegó primero' NO es necesariamente el que se atiende primero."*

## El ADT Priority Queue (Max-PQ)

```
insert(key)   → insertar un elemento con su prioridad
delMax()      → remover y retornar el elemento de mayor prioridad
max()         → consultar el elemento de mayor prioridad (sin remover)
isEmpty()     → ¿está vacía la cola?
size()        → número de elementos
```

La versión **Min-PQ** (con `delMin` y `min`) es simétrica — útil para Dijkstra.

## ¿Por qué no usar una lista o arreglo ordenado?

| Implementación | insert | delMax | Observación |
|---------------|--------|--------|-------------|
| Arreglo desordenado | O(1) | O(n) | Hay que buscar el máximo |
| Arreglo ordenado | O(n) | O(1) | Insertar desplaza elementos |
| **Binary Heap** | **O(log n)** | **O(log n)** | El punto dulce |

> 📌 El heap no es la mejor estructura para ninguna operación individual, pero es la mejor en equilibrio para ambas operaciones juntas.

# Sección 2: El Árbol Binario Completo (8 minutos)

## Definición

Un **árbol binario completo** es un árbol binario donde:
- Todos los niveles están llenos, excepto posiblemente el último.
- El último nivel se llena de **izquierda a derecha**.

```
           T
         /   \
        S     R
       / \   / \
      P   N O   A
     / \
    E   I
```

## La propiedad heap-order

Un árbol binario completo satisface la **propiedad heap-order** si:

> La clave de cada nodo es **mayor o igual** a las claves de sus hijos.

Consecuencia directa: **la raíz siempre contiene el máximo**.

```
           T          ← raíz = máximo
         /   \
        S     R       ← S ≤ T, R ≤ T
       / \   / \
      P   N O   A     ← P ≤ S, N ≤ S, O ≤ R, A ≤ R
     / \
    E   I             ← E ≤ P, I ≤ P
```

> 🎙️ **[PAUSA PROFESOR]** *"¿Qué relación hay entre P y R en este árbol heap? ¿Pueden compararse?"*  
> *Respuesta esperada: No hay relación garantizada entre nodos de distintas ramas — solo se garantiza la relación padre-hijo.*

## Altura de un árbol binario completo

Para n nodos, la altura es **⌊log₂ n⌋**.

| n | altura |
|---|--------|
| 1 | 0 |
| 3 | 1 |
| 7 | 2 |
| 15 | 3 |
| 1,000,000 | 19 |
| 1,000,000,000 | 29 |

Esta altura acotada es el origen de la complejidad O(log n) del heap.

# Sección 3: Representación como Arreglo (7 minutos)

## El truco del heap: árbol en un arreglo

Un árbol binario completo se puede representar en un arreglo con indexación **1-based** (índice 1 = raíz), aprovechando que el árbol es completo.

```
Árbol:                    Arreglo (índice 1-based):
       T(1)               índice: 1  2  3  4  5  6  7  8  9
      /    \              valor:  T  S  R  P  N  O  A  E  I
    S(2)   R(3)
   /   \   /  \
 P(4) N(5) O(6) A(7)
 /  \
E(8) I(9)
```

### Las relaciones índice-nodo

Para un nodo en posición **k**:
- **Padre:** `k // 2`
- **Hijo izquierdo:** `2 * k`
- **Hijo derecho:** `2 * k + 1`

```python
# Ejemplo: nodo en posición 4 (P)
k = 4
padre       = k // 2     # = 2 → S ✓
hijo_izq    = 2 * k      # = 8 → E ✓
hijo_der    = 2 * k + 1  # = 9 → I ✓
```

> 📌 No se necesitan punteros, ni nodos con referencias. Un simple arreglo es suficiente. Esta es la razón por la que los heaps son tan eficientes en la práctica.

In [ ]:
# Visualización del árbol binario completo como arreglo
def visualizar_heap(heap, titulo="Heap"):
    """
    Visualiza un heap (arreglo 1-based) como árbol binario.
    heap[0] se ignora, heap[1] es la raíz.
    """
    n = len(heap) - 1  # número de elementos reales
    if n == 0:
        print("Heap vacío")
        return

    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.1, 1.1)
    ax.axis('off')
    ax.set_title(titulo, fontsize=14, fontweight='bold')

    altura = int(np.floor(np.log2(n))) + 1
    posiciones = {}

    def calcular_posicion(k, nivel, izq, der):
        if k > n:
            return
        x = (izq + der) / 2
        y = 1 - nivel * (1 / altura)
        posiciones[k] = (x, y)
        calcular_posicion(2*k, nivel+1, izq, x)
        calcular_posicion(2*k+1, nivel+1, x, der)

    calcular_posicion(1, 0, 0, 1)

    # Dibujar aristas
    for k in range(1, n+1):
        if 2*k <= n:
            x1, y1 = posiciones[k]
            x2, y2 = posiciones[2*k]
            ax.plot([x1, x2], [y1, y2], 'k-', linewidth=1.5, zorder=1)
        if 2*k+1 <= n:
            x1, y1 = posiciones[k]
            x2, y2 = posiciones[2*k+1]
            ax.plot([x1, x2], [y1, y2], 'k-', linewidth=1.5, zorder=1)

    # Dibujar nodos
    for k in range(1, n+1):
        x, y = posiciones[k]
        color = '#FF6B6B' if k == 1 else '#4ECDC4'
        circle = plt.Circle((x, y), 0.04, color=color, zorder=2)
        ax.add_patch(circle)
        ax.text(x, y, str(heap[k]), ha='center', va='center',
                fontsize=10, fontweight='bold', color='white', zorder=3)
        ax.text(x, y - 0.08, f'[{k}]', ha='center', va='top',
                fontsize=7, color='gray', zorder=3)

    plt.tight_layout()
    plt.show()

# Ejemplo del árbol de la Sección 2
heap_ejemplo = [None, 'T', 'S', 'R', 'P', 'N', 'O', 'A', 'E', 'I']
visualizar_heap(heap_ejemplo, "Max-Heap con letras (raíz en rojo = máximo)")

# Verificar las relaciones índice-nodo
print("Verificación de relaciones para k=4 (P):")
k = 4
print(f"  heap[{k}] = {heap_ejemplo[k]}")
print(f"  Padre   heap[{k//2}] = {heap_ejemplo[k//2]}")
print(f"  H. Izq  heap[{2*k}]  = {heap_ejemplo[2*k]}")
print(f"  H. Der  heap[{2*k+1}]  = {heap_ejemplo[2*k+1]}")

# Sección 4: swim — Restaurar el Heap hacia Arriba (10 minutos)

## ¿Cuándo se viola la propiedad heap?

Al **insertar** un nuevo elemento, lo ponemos al final del arreglo (siguiente posición libre). Esto puede violar la propiedad heap-order si el nuevo elemento es mayor que su padre.

## La operación swim (nadar hacia arriba)

```
Heap válido:              Insertamos 'T' al final:
      S                        S
    /   \                    /   \
   P     R       →          P     R
  / \   /                  / \   / \
 E   I O                  E   I O   T    ← T > R: viola heap!

swim(T): comparar con padre, intercambiar si T > padre

Paso 1: T > R → swap(T, R)
      S
    /   \
   P     T       ← T > S?: Sí → swap(T, S)
  / \   / \
 E   I O   R

Paso 2: T > S → swap(T, S)
      T          ← T ≥ todos sus hijos ✓
    /   \
   P     S
  / \   / \
 E   I O   R
```

**Invariante:** En cada paso, el subárbol desde la posición actual hacia abajo ya satisface heap-order. Solo puede haber violación con el padre.

> 🎙️ **[PAUSA PROFESOR]** *"¿Cuántos intercambios puede requerir swim en el peor caso? ¿De qué depende ese número?"*  
> *Respuesta esperada: ⌊log₂ n⌋ intercambios — la altura del árbol.*

In [ ]:
class MaxHeap:
    """
    Max-Heap con indexación 1-based.
    heap[0] no se usa (es None).
    """
    def __init__(self):
        self.heap = [None]   # posición 0 sin usar
        self.n = 0           # número de elementos

    def size(self):
        return self.n

    def isEmpty(self):
        return self.n == 0

    def _swap(self, i, j):
        self.heap[i], self.heap[j] = self.heap[j], self.heap[i]

    def _less(self, i, j):
        """¿Es heap[i] < heap[j]?"""
        return self.heap[i] < self.heap[j]

    # ─────────────────────────────────────────────
    # swim: restaurar heap hacia arriba
    # ─────────────────────────────────────────────
    def _swim(self, k):
        """
        Mueve el elemento en posición k hacia arriba
        hasta que la propiedad heap-order se restaure.
        Condición: k > 1 (no es la raíz) Y heap[k] > heap[padre(k)]
        """
        while k > 1 and self._less(k // 2, k):
            self._swap(k // 2, k)   # intercambiar con el padre
            k = k // 2              # subir al padre

    def insert(self, key):
        """
        Insertar: agregar al final y nadar hacia arriba.
        Complejidad: O(log n)
        """
        self.heap.append(key)   # agregar al final
        self.n += 1
        self._swim(self.n)      # restaurar heap-order

    def max(self):
        """El máximo siempre está en la raíz (posición 1)."""
        if self.isEmpty():
            raise IndexError("Heap vacío")
        return self.heap[1]

    def __repr__(self):
        return f"MaxHeap{self.heap[1:]}"


# Demo de inserción
pq = MaxHeap()
elementos = ['P', 'R', 'N', 'O', 'A', 'T', 'S']

print("Insertando elementos uno a uno:")
print("-" * 45)
for e in elementos:
    pq.insert(e)
    print(f"  insert('{e}') → heap = {pq.heap[1:]:}  máx = {pq.max()}")

print(f"\nHeap final: {pq}")
visualizar_heap(pq.heap, f"Max-Heap tras insertar {elementos}")

## Tracing de swim paso a paso

In [ ]:
def swim_verbose(heap, k):
    """Versión pedagógica de swim que imprime cada paso."""
    print(f"swim({heap[k]}) desde posición {k}")
    pasos = 0
    while k > 1 and heap[k // 2] < heap[k]:
        padre = k // 2
        print(f"  heap[{k}]={heap[k]} > heap[{padre}]={heap[padre]} → swap")
        heap[k], heap[padre] = heap[padre], heap[k]
        k = padre
        pasos += 1
    print(f"  heap[{k}]={heap[k]} ≤ padre (o es raíz) → stop")
    print(f"  Total intercambios: {pasos}")
    return heap

# Demostración: insertamos 'T' (peor caso — debe subir hasta la raíz)
heap_demo = [None, 'S', 'P', 'R', 'E', 'I', 'O', None]  # sin posición 7 aún
heap_demo[7] = 'T'   # insertar T al final
n_demo = 7
print(f"Estado antes de swim: {heap_demo[1:]}")
print()
swim_verbose(heap_demo, n_demo)
print(f"\nEstado después de swim: {heap_demo[1:]}")

# Sección 5: sink — Restaurar el Heap hacia Abajo (10 minutos)

## ¿Cuándo usamos sink?

Al ejecutar **delMax** (extraer el máximo), necesitamos:
1. Guardar `heap[1]` (la raíz = máximo) para retornarlo.
2. Mover el último elemento a la raíz (para mantener la forma completa del árbol).
3. Restaurar la propiedad heap-order con **sink**.

## La operación sink (hundir hacia abajo)

```
delMax del heap:     Mover último a raíz:    sink(A):
      T                    A                 A < max(S,R)=S → swap(A,S)
    /   \                /   \                      S
   S     R    →        S     R    →               /   \
  / \   / \           / \   /                    A     R
 P   N O   A         P   N O              →  A < max(P,N)=P → swap(A,P)
                                                    S
                                                  /   \
                                                 P     R
                                                / \
                                               A   N   ← A ≥ hijos (ninguno) ✓
```

**Invariante de sink:** En cada paso, el elemento actual puede ser menor que alguno de sus hijos. Se intercambia con el **hijo mayor** (no con cualquier hijo — el hijo mayor es el candidato a ocupar la posición del padre).

> 🎙️ **[PAUSA PROFESOR]** *"¿Por qué intercambiamos con el hijo MAYOR y no con cualquier hijo que sea mayor que nosotros?"*  
> *Respuesta esperada: Si intercambiamos con el hijo menor, el hijo mayor quedaría por encima del hijo menor violando la propiedad heap.*

In [ ]:
# Completar la clase MaxHeap con sink y delMax

class MaxHeap:
    def __init__(self):
        self.heap = [None]
        self.n = 0

    def size(self):    return self.n
    def isEmpty(self): return self.n == 0

    def _swap(self, i, j):
        self.heap[i], self.heap[j] = self.heap[j], self.heap[i]

    def _less(self, i, j):
        return self.heap[i] < self.heap[j]

    def _swim(self, k):
        while k > 1 and self._less(k // 2, k):
            self._swap(k // 2, k)
            k = k // 2

    # ─────────────────────────────────────────────
    # sink: restaurar heap hacia abajo
    # ─────────────────────────────────────────────
    def _sink(self, k):
        """
        Mueve el elemento en posición k hacia abajo
        hasta que la propiedad heap-order se restaure.
        """
        while 2 * k <= self.n:          # mientras tenga al menos un hijo
            j = 2 * k                   # j = hijo izquierdo
            if j < self.n and self._less(j, j+1):
                j += 1                  # j = hijo derecho (si es mayor)
            if not self._less(k, j):    # ¿ya está en su lugar?
                break
            self._swap(k, j)            # intercambiar con el hijo mayor
            k = j                       # bajar al hijo

    def insert(self, key):
        self.heap.append(key)
        self.n += 1
        self._swim(self.n)

    def delMax(self):
        """
        Remover y retornar el máximo.
        Complejidad: O(log n)
        """
        if self.isEmpty():
            raise IndexError("Heap vacío")
        maximo = self.heap[1]               # guardar el máximo (raíz)
        self._swap(1, self.n)               # mover el último a la raíz
        self.heap.pop()                     # eliminar el último
        self.n -= 1
        if not self.isEmpty():
            self._sink(1)                   # restaurar heap-order
        return maximo

    def max(self):
        if self.isEmpty(): raise IndexError("Heap vacío")
        return self.heap[1]

    def __repr__(self):
        return f"MaxHeap{self.heap[1:]}"


# Demo completa: insertar y extraer
pq = MaxHeap()
for v in [5, 3, 8, 1, 7, 2, 9, 4, 6]:
    pq.insert(v)

print(f"Heap inicial: {pq}")
visualizar_heap(pq.heap, "Max-Heap con enteros")

print("\nExtrayendo elementos en orden:")
extraidos = []
while not pq.isEmpty():
    val = pq.delMax()
    extraidos.append(val)
    print(f"  delMax() = {val}  → heap = {pq.heap[1:]}")

print(f"\nOrden de extracción: {extraidos}")
print("✅ Los elementos salen en orden decreciente — esto es Heapsort!")

# Sección 6: Análisis de Complejidad (5 minutos)

## ¿Por qué O(log n)?

| Operación | Descripción | Complejidad |
|-----------|-------------|-------------|
| `insert` | Agregar al final + swim hasta la raíz | O(log n) |
| `delMax` | Swap raíz-último + sink hasta hoja | O(log n) |
| `max` | Leer heap[1] | O(1) |
| `size` | Leer self.n | O(1) |

Ambas operaciones recorren como máximo la **altura del árbol** (⌊log₂ n⌋), que es el número máximo de intercambios posibles.

## Número de comparaciones

- **swim:** ≤ ⌊log₂ n⌋ comparaciones (una por nivel).
- **sink:** ≤ 2⌊log₂ n⌋ comparaciones (dos por nivel: comparar hermanos + comparar con padre).

In [ ]:
# Análisis empírico: comparaciones vs n
import random

class MaxHeapInstrumentado(MaxHeap):
    """MaxHeap que cuenta comparaciones e intercambios."""
    def __init__(self):
        super().__init__()
        self.comparaciones = 0
        self.intercambios  = 0

    def _less(self, i, j):
        self.comparaciones += 1
        return self.heap[i] < self.heap[j]

    def _swap(self, i, j):
        self.intercambios += 1
        super()._swap(i, j)

ns = [100, 500, 1000, 5000, 10000, 50000]
resultados_insert = []
resultados_delmax = []

for n in ns:
    datos = random.sample(range(n * 10), n)

    # Medir insert
    pq = MaxHeapInstrumentado()
    for v in datos:
        pq.insert(v)
    resultados_insert.append(pq.comparaciones / n)  # comparaciones por inserción

    # Medir delMax (sobre el heap ya construido)
    pq2 = MaxHeapInstrumentado()
    for v in datos:
        pq2.insert(v)
    pq2.comparaciones = 0  # resetear
    pq2.intercambios  = 0
    for _ in range(n):
        pq2.delMax()
    resultados_delmax.append(pq2.comparaciones / n)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

teorico = [np.log2(n) for n in ns]

axes[0].plot(ns, resultados_insert, 'bo-', label='Empírico (insert)', linewidth=2)
axes[0].plot(ns, teorico, 'r--', label='log₂(n) teórico', linewidth=2)
axes[0].set_xlabel('n')
axes[0].set_ylabel('Comparaciones por operación')
axes[0].set_title('insert: comparaciones / n')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

teorico2 = [2 * np.log2(n) for n in ns]
axes[1].plot(ns, resultados_delmax, 'go-', label='Empírico (delMax)', linewidth=2)
axes[1].plot(ns, teorico2, 'r--', label='2·log₂(n) teórico', linewidth=2)
axes[1].set_xlabel('n')
axes[1].set_ylabel('Comparaciones por operación')
axes[1].set_title('delMax: comparaciones / n')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Binary Heap — Análisis Empírico de Complejidad', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"{'n':>8} {'Comp/insert':>14} {'log₂n':>10} {'Comp/delMax':>14} {'2·log₂n':>10}")
print("-" * 60)
for i, n in enumerate(ns):
    print(f"{n:>8} {resultados_insert[i]:>14.2f} {np.log2(n):>10.2f} {resultados_delmax[i]:>14.2f} {2*np.log2(n):>10.2f}")

# Sección 7: Resumen y Conexión con la Próxima Clase (5 minutos)

## Lo que aprendimos hoy

| Concepto | Descripción |
|---------|-------------|
| **Priority Queue ADT** | Cola donde el máximo siempre tiene prioridad de salida |
| **Árbol binario completo** | Todos los niveles llenos excepto el último (de izq a der) |
| **Heap-order** | Padre ≥ hijos en todo el árbol |
| **Representación en arreglo** | Nodo k → padre k//2, hijos 2k y 2k+1 |
| **swim** | Restaurar heap desde abajo hacia arriba (para insert) |
| **sink** | Restaurar heap desde arriba hacia abajo (para delMax) |
| **Complejidad** | insert: O(log n), delMax: O(log n), max: O(1) |

## La observación clave de la demo

Cuando ejecutamos `delMax()` repetidamente sobre un heap, los elementos **salen en orden decreciente**.  
Esto no es casualidad: es la base de **Heapsort**.

> 🎙️ **[PREGUNTA DE CIERRE]** *"Vimos que extraer todos los elementos del heap los ordena de mayor a menor. ¿Cómo convertiríamos esto en un algoritmo de ordenamiento in-place? ¿Cuál es el problema con el enfoque directo?"*

*(Respuesta en la Clase 7 — Heapsort)*

---

## Referencias

- Sedgewick & Wayne. *Algorithms*, 4ª ed., Sección 2.4
- Williams, J.W.J. (1964). "Algorithm 232: Heapsort". *Communications of the ACM*, 7(6), 347–348.
- Cormen et al. *Introduction to Algorithms* (CLRS), Cap. 6